# Charts

Seven short lessons, each on one real dataset from [The Pudding](https://pudding.cool), taken
straight from [their public data repo](https://github.com/the-pudding/data).

1. A bar, for comparing two things
2. Bars hide the spread
3. A line, for change over time
4. Divide by whatever grew anyway
5. When the colour is the data
6. The category is a choice
7. The title is the finding

No new library. `pandas` to hold the numbers, `matplotlib` to draw them.

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import requests

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "font.size": 10})
RUST, BLUE, GREY = "#A34526", "#2E6E8E", "#BBB5AE"

PUDDING = "https://raw.githubusercontent.com/the-pudding/data/master/"

def pudding(path):
    """Read one of The Pudding's CSVs straight off GitHub."""
    return pd.read_csv(io.StringIO(requests.get(PUDDING + path, timeout=60).text))

## 1 · A bar, for comparing two things

*Women's Pockets are Inferior* (2018). Someone measured the pockets in 80 pairs of jeans.

In [ ]:
jeans = pudding("pockets/measurements.csv")
print(len(jeans), "pairs,", jeans["brand"].nunique(), "brands")
jeans[["brand", "style", "menWomen", "maxHeightFront", "maxWidthFront"]].head()

In [ ]:
depth = jeans.groupby("menWomen")["maxHeightFront"].mean()
print(depth.round(1))

fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.bar(["men's", "women's"], [depth["men"], depth["women"]], color=[BLUE, RUST], width=0.55)
for i, v in enumerate([depth["men"], depth["women"]]):
    ax.text(i, v + 0.4, f"{v:.1f}", ha="center")
ax.set_ylabel("front pocket depth (cm)")
ax.set_title("Women's jeans have shallower pockets")
plt.tight_layout(); plt.show()

Two numbers, two bars. The bars start at zero, so twice as tall means twice as deep.

## 2 · Bars hide the spread

Those two bars are 80 pairs of jeans squashed into two numbers. Draw all 80.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for i, (group, colour) in enumerate([("men", BLUE), ("women", RUST)]):
    vals = jeans.loc[jeans["menWomen"] == group, "maxHeightFront"]
    ax.scatter(vals, [i] * len(vals), color=colour, alpha=0.6, s=45)
    ax.scatter(vals.mean(), i, color="black", marker="|", s=400)
ax.set_yticks([0, 1], ["men's", "women's"])
ax.set_ylim(-0.7, 1.7)
ax.set_xlabel("front pocket depth (cm)")
ax.set_title("Every pair, with the average marked")
plt.tight_layout(); plt.show()

shallowest_men = jeans.loc[jeans["menWomen"] == "men", "maxHeightFront"].min()
deeper = (jeans.loc[jeans["menWomen"] == "women", "maxHeightFront"] > shallowest_men).sum()
print(f"{deeper} of 40 women's pairs beat the shallowest men's pair ({shallowest_men} cm)")

The two groups barely touch. One pair out of forty. The bar chart was true but the dots are the
argument.

## 3 · A line, for change over time

*The Names in Songs* — every first name sung on the Billboard Hot 100, 1958 to 2019.

In [ ]:
songs = pudding("names-in-songs/unique.csv")
songs["year"] = pd.to_numeric(songs["year"], errors="coerce")
people = songs[songs["person"] == True]
people = people[people["year"] < 2019]          # 2019 is a part-year, it would dip for no reason
print(len(people), "name mentions,", int(people["year"].min()), "to", int(people["year"].max()))
people[["artist", "song", "name", "year"]].head()

In [ ]:
per_year = people.groupby("year").size()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(per_year.index, per_year.values, color=RUST, lw=2)
ax.set_ylabel("names sung")
ax.set_title("Names in hit songs, per year")
plt.tight_layout(); plt.show()
print("1960:", per_year[1960], " 2018:", per_year[2018])

Ten times as many names as in 1960. Believe it?

(The last year in the file stops mid-year, so it is dropped. A half-year plotted next to full
ones looks like a crash.)

## 4 · Divide by whatever grew anyway

The dataset has more songs in it every year too. So of course it has more names.

In [ ]:
counted = pd.DataFrame({"names": people.groupby("year").size(),
                        "songs": people.groupby("year")["song"].nunique()})
counted["per_song"] = counted["names"] / counted["songs"]

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
a.plot(counted.index, counted["names"], color=RUST, lw=2)
a.set_title("Names, counted")
b.plot(counted.index, counted["per_song"], color=BLUE, lw=2)
b.set_ylim(0, 2.6)
b.set_title("Names per song")
plt.tight_layout(); plt.show()

print(counted.loc[[1960, 1980, 2000, 2018]].round(2).to_string())

The left chart rises tenfold. The right one goes from about 1.5 to about 2.2. Most of the first
chart was the dataset growing, not songwriting changing.

Ask this of every count you plot: what else got bigger?

## 5 · When the colour is the data

*The Naked Truth* — 625 foundation shades from 36 makeup brands, each with its own hex code.

In [ ]:
shades = pudding("makeup-shades/shades.csv")
print(len(shades), "shades,", shades["brand"].nunique(), "brands")
shades[["brand", "product", "hex", "L"]].head()

In [ ]:
# L is lightness, 0 darkest to 100 lightest. Draw each shade in its own colour.
BRANDS = ["Fenty", "Maybelline", "MAC", "Estée Lauder", "L'Oréal"]
fig, ax = plt.subplots(figsize=(7.5, 3))
for i, brand in enumerate(BRANDS):
    rows = shades[shades["brand"] == brand].sort_values("L")
    ax.scatter(rows["L"], [i] * len(rows), c="#" + rows["hex"], s=90,
               edgecolor="#444", linewidth=0.4)
ax.set_yticks(range(len(BRANDS)), BRANDS)
ax.set_xlabel("lightness (0 darkest, 100 lightest)")
ax.set_title("Every shade, in its own colour")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

The marks are the data. No legend needed, because nothing had to be translated into a key.

Colour earns its place when it *is* the measurement. The rest of the time a second colour is
usually one more thing to decode.

## 6 · The category is a choice

Back to the songs. Which names get sung most?

In [ ]:
top = people["name"].value_counts().head(10)
print(top.to_string())

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(top.index[::-1], top.values[::-1], color=GREY)
ax.set_title("Most-sung names")
plt.tight_layout(); plt.show()

"Baby" is 23% of every name in the dataset, and it is not a name. Somebody decided it counted.

Drop it and the chart is about something else.

In [ ]:
real = top.drop(["Baby", "Jesus"])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(real.index[::-1], real.values[::-1], color=RUST)
ax.set_title("Most-sung names, without Baby and Jesus")
plt.tight_layout(); plt.show()

Neither chart is wrong. They answer different questions, and the difference is a decision
somebody made before any code ran. Write that decision down.

## 7 · The title is the finding

Same numbers, three titles. Only one of them tells the reader what to see.

In [ ]:
depth = jeans.groupby("menWomen")["maxHeightFront"].mean()
TITLES = ["Figure 3", "Front pocket depth by gender",
          "Women's jean pockets are 38% shallower"]

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
for ax, title in zip(axes, TITLES):
    ax.bar(["men's", "women's"], [depth["men"], depth["women"]], color=[BLUE, RUST], width=0.55)
    ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

print(f"{100 * (1 - depth['women'] / depth['men']):.0f}% shallower")

The third one is the only one doing any work. Write the title last, once you know what the
chart says, and make it a sentence.

## Your turn

The Pudding's [data repo](https://github.com/the-pudding/data) has about forty datasets. Pick
one and make two charts from it: the obvious one, and one that complicates it.

```python
df = pudding("dress-codes/banned_items.csv")     # or vogue/models.csv, births/births.csv
df.head()
```

Four things to check before you show anyone a chart:

1. Does the bar start at zero? If not, say why.
2. Have you hidden a spread inside an average?
3. Did the count go up, or did the dataset?
4. Does the title say what you found?